# Финальная оценка ALS и холодного старта

Параметры ALS выбираются по среднему warm-validation NDCG@10 в опыте 08;
seed=42 зафиксирован заранее. Переобучаем на train+validation, заново применяя
фильтр 5/5 и веса. Сравниваем ALS с популярностью за 28 дней и за всё время.
Warm — минимум 5 записей полной предшествующей истории. Цели test не фильтруем.
ALS оценивается без fallback: неизвестные пользователи получают пустой список.

Для cold проверяем уже выбранное в опыте 02 правило: без истории — популярность
за всё время; 1–4 записи — жанровая популярность за 28 дней, beta=1.
При отсутствии положительного жанрового профиля этот алгоритм возвращает обычное
популярное за 28 дней. Это техническое правило ранее не было проверено на отдельной
группе validation. Сравниваем с обоими baseline, параметры по test не меняем.

**Ограничения:** период test ранее изучался в EDA и не является полностью
независимым. updated_at — время последнего изменения записи, NaT считаются старыми;
жанры получены из текущего API, а не исторического снимка. Для строгой проверки
нужен новый снимок. Метрики validation и test относятся к разным периодам и группам.


In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import sys, json, hashlib
from pathlib import Path
from time import perf_counter
import importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display
sys.path.insert(0, str(Path.cwd().parent))
from app.training.datasets import load_anilist, filter_interactions
from app.training.evaluate import chronological_split, relevant_items, evaluate
from app.training.weights import weighted_interactions
from app.training.anilist_genres import load_item_genres, DEFAULT_GENRES_PATH
from app.recsys.als import ALSRecommender
from app.recsys.popularity import PopularityRecommender, GenrePopularityRecommender
pd.set_option("display.float_format", "{:,.6f}".format)
root = Path.cwd().parent
selection_path = root / "artifacts/als_combined_parameters/protocol.json"
selection = json.loads(selection_path.read_text())
assert selection["test_evaluated"] is False
assert selection["final_seed"] == 42
params = {**selection["selected_parameters"], "random_state": 42}
out = root / "artifacts/final_evaluation"
out.mkdir(parents=True, exist_ok=True)
# Записываем зафиксированный протокол до загрузки test и вычисления его метрик.
protocol = {
    "selection_source": str(selection_path.relative_to(root)),
    "selection_sha256": hashlib.sha256(selection_path.read_bytes()).hexdigest(),
    "selection_metric": "mean warm-validation ndcg@10",
    "als_parameters": params, "score_coef": 0.0,
    "status_weights": selection["status_weights"], "min_score": 5,
    "half_life_days": None, "min_user_items": 5, "min_item_users": 5,
    "validation_start": selection["validation_start"], "test_start": selection["test_start"],
    "snapshot_end": selection["snapshot_end"], "training": "train + validation",
    "target_filtering": False, "exclusion_history": "full train + validation",
    "warm_fallback": None, "cold_no_history": "all-time popularity",
    "cold_short_history": "genre popularity 28 days, beta=1; plain 28 days without profile",
    "test_independence": "limited: period previously explored in EDA",
    "id_namespace": "anilist", "confidence": "1 + alpha * weight",
    "versions": {x: importlib.metadata.version(x) for x in ["implicit", "numpy", "scipy", "pandas"]},
    "genres_sha256": hashlib.sha256(DEFAULT_GENRES_PATH.read_bytes()).hexdigest(),
}
(out / "protocol.json").write_text(json.dumps(protocol, ensure_ascii=False, indent=2) + "\n")
print("Зафиксированы параметры:", params)


Зафиксированы параметры: {'factors': 64, 'regularization': 0.1, 'alpha': 5, 'iterations': 20, 'num_threads': 4, 'random_state': 42}


/home/deonik3/AniFolder/ml/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
inter = load_anilist()
assert inter.updated_at.max() == pd.Timestamp(protocol["snapshot_end"])
train, validation, test = chronological_split(inter,
    validation_start=pd.Timestamp(protocol["validation_start"]),
    test_start=pd.Timestamp(protocol["test_start"]))
history = pd.concat([train, validation], ignore_index=True)
assert len(history) == 3513393 + 218925 and len(test) == 392661
assert history.updated_at.dropna().lt(pd.Timestamp(protocol["test_start"])).all()
assert not history.duplicated(["user_id", "anime_id"]).any()
training = filter_interactions(history, min_user_items=5, min_item_users=5)
weighted = weighted_interactions(training, status_weights=protocol["status_weights"],
    score_coef=0.0, min_score=5, half_life_days=None)
assert not weighted.status.eq("dropped").any()
assert not weighted.score.lt(5).fillna(False).any()
targets = relevant_items(test)
counts = history.user_id.value_counts()
groups = {
    "warm": {u: t for u, t in targets.items() if counts.get(u, 0) >= 5},
    "no_history": {u: t for u, t in targets.items() if counts.get(u, 0) == 0},
    "short_history": {u: t for u, t in targets.items() if 0 < counts.get(u, 0) < 5},
}
assert sum(map(len, groups.values())) == len(targets)
seen = history.groupby("user_id").anime_id.agg(set).to_dict()
print("История / после фильтра / положительные пары:", len(history), len(training), len(weighted))
display(pd.Series({g: len(t) for g, t in groups.items()}, name="users"))


История / после фильтра / положительные пары: 3732318 3719273 3505358


warm             8561
no_history       1110
short_history     312
Name: users, dtype: int64

## Переобучение и warm-test

In [3]:
started = perf_counter()
als = ALSRecommender(**params).fit(weighted, history=history)
fit_seconds = perf_counter() - started
warm_users = sorted(groups["warm"])
als_recs = als.recommend_many(warm_users, k=10)
now = pd.Timestamp(protocol["test_start"])
pop28 = PopularityRecommender(window_days=28).fit(training, now=now, history=history)
pop_all = PopularityRecommender(window_days=None).fit(training, now=now, history=history)
pop28_recs = pop28.recommend_many(sorted(targets), k=10)
pop_all_recs = pop_all.recommend_many(sorted(targets), k=10)
records = []
def record(name, group, recs):
    goal = groups[group]
    recs = {u: recs[u] for u in goal}
    assert all(len(r) <= 10 and len(r) == len(set(r)) and not (set(r) & seen.get(u, set()))
               for u, r in recs.items())
    records.append({"model": name, "group": group, **evaluate(recs, goal, k=10),
                    "short_lists": sum(len(r) < 10 for r in recs.values())})
record("ALS", "warm", als_recs)
record("popularity28", "warm", pop28_recs)
record("popularity_all", "warm", pop_all_recs)
print("Обучение, секунд:", fit_seconds)
print("Warm-пользователей вне ALS:", sum(u not in als.user_index_ for u in warm_users))
catalog_ids = set(als.item_ids_)
print("Доля warm-целей вне каталога ALS:",
      sum(len(t - catalog_ids) for t in groups["warm"].values()) / sum(map(len, groups["warm"].values())))
display(pd.DataFrame(records).set_index(["group", "model"]))


Обучение, секунд: 7.57914426800005
Warm-пользователей вне ALS: 0
Доля warm-целей вне каталога ALS: 0.01583801791319106


precision@10  recall@10  ndcg@10  coverage@10  users  \
group model                                                                  
warm  ALS                 0.136853   0.066534 0.160402         3176   8561   
      popularity28        0.094568   0.046319 0.121586           80   8561   
      popularity_all      0.069852   0.028674 0.079160          316   8561   

                      short_lists  
group model                        
warm  ALS                       0  
      popularity28              0  
      popularity_all            0

## Cold-test с ранее выбранным правилом

In [4]:
cold_users = sorted(set(groups["no_history"]) | set(groups["short_history"]))
cold_history = history[history.user_id.isin(cold_users)]
genre = GenrePopularityRecommender(load_item_genres(), window_days=28, beta=1).fit(
    training, now=now, history=cold_history)
genre_recs = genre.recommend_many(groups["short_history"], k=10)
policy_recs = {u: pop_all_recs[u] if u in groups["no_history"] else genre_recs[u] for u in cold_users}
for group in ["no_history", "short_history"]:
    record("selected_cold_policy", group, policy_recs)
    record("popularity28", group, pop28_recs)
    record("popularity_all", group, pop_all_recs)
print("Из 1–4 записей без положительного жанрового профиля:",
      sum(u not in genre.preferences_ for u in groups["short_history"]))
results = pd.DataFrame(records)
display(results.set_index(["group", "model"]))


Из 1–4 записей без положительного жанрового профиля: 1


precision@10  recall@10  ndcg@10  \
group         model                                                    
warm          ALS                       0.136853   0.066534 0.160402   
              popularity28              0.094568   0.046319 0.121586   
              popularity_all            0.069852   0.028674 0.079160   
no_history    selected_cold_policy      0.182523   0.039449 0.198957   
              popularity28              0.075586   0.017833 0.090716   
              popularity_all            0.182523   0.039449 0.198957   
short_history selected_cold_policy      0.059295   0.042732 0.089762   
              popularity28              0.056731   0.041521 0.086026   
              popularity_all            0.077885   0.031905 0.088049   

                                    coverage@10  users  short_lists  
group         model                                                  
warm          ALS                          3176   8561            0  
              popularity28                   80   8561            0  
              popularity_all                316   8561            0  
no_history    selected_cold_policy           10   1110            0  
              popularity28                   10   1110            0  
              popularity_all                 10   1110            0  
short_history selected_cold_policy           24    312            0  
              popularity28                   13    312            0  
              popularity_all                 12    312            0

## Сохранение модели и результатов

In [5]:
artifact = out / "als_final.npz"
als.save(artifact, metadata=protocol)
loaded = ALSRecommender.load(artifact)
assert loaded.params == params and loaded.metadata_ == protocol
assert loaded.recommend_many(warm_users, k=10) == als_recs
results.to_csv(out / "metrics.csv", index=False)
(out / "data_summary.json").write_text(json.dumps({
    "history_rows": len(history), "filtered_rows": len(training), "positive_rows": len(weighted),
    "als_users": len(als.user_ids_), "als_items": len(als.item_ids_),
    "test_rows": len(test), "groups": {g: len(t) for g, t in groups.items()},
    "fit_seconds": fit_seconds, "test_evaluated": True, "roundtrip_verified": True,
}, indent=2) + "\n")
print("Сохранено:", artifact)
print("Выдача после загрузки точно совпадает для всех warm-пользователей.")


Сохранено: /home/deonik3/AniFolder/ml/artifacts/final_evaluation/als_final.npz
Выдача после загрузки точно совпадает для всех warm-пользователей.


## Вывод

Параметры, выбранные до оценки test: **factors=64, alpha=5,
regularization=0,1, iterations=20, num_threads=4, seed=42**;
score_coef=0, planned=0,5, фильтр 5/5, без затухания, оценки <5 исключены.
После переобучения: 3 505 358 положительных пар, 8564 пользователя,
16 045 тайтлов. Полная история исключений — 3 732 318 записей.

### Пользователи с историей от 5 записей (8561)

| Модель | NDCG@10 | Precision@10 | Recall@10 | Coverage@10 |
|---|---:|---:|---:|---:|
| ALS | 0.160402 | 0.136853 | 0.066534 | 3176 |
| Популярность 28 дней | 0.121586 | 0.094568 | 0.046319 | 80 |
| Популярность за всё время | 0.079160 | 0.069852 | 0.028674 | 316 |

На этом периоде ALS выше популярности 28 дней по NDCG на 0,038816
(**+31,9%**), а также по precision и recall. Все warm-пользователи известны
модели; все получили полный топ-10. Около 1,58% релевантных warm-целей вне
каталога ALS остались в оценке. Coverage — число разных рекомендованных тайтлов,
само по себе оно не доказывает качество персонализации.

### Холодный старт

- **Без истории (1110 пользователей):** ранее выбранная популярность за всё
  время даёт NDCG=0,198957 против 0,090716 у популярности 28 дней.
- **1–4 записи (312 пользователей):** выбранный genre28, beta=1 даёт
  NDCG=0,089762 против 0,086026 у plain28 и 0,088049 у all-time.
  Recall соответственно 0,042732 / 0,041521 / 0,031905; coverage 24 / 13 / 12.
  Выигрыш NDCG небольшой, статистическая значимость не проверялась.
  У одного пользователя нет положительного жанрового профиля: сработал
  предусмотренный алгоритмом переход к plain28. Отдельно качество этого
  случая на одном пользователе обобщать нельзя.

Все оценённые варианты выдали полный топ-10 без дублей и пересечений с полной
предшествующей историей. Сохранение и загрузка `als_final.npz` точно сохраняют
выдачу для всех 8561 warm-пользователей. Артефакты и CSV — в
`ml/artifacts/final_evaluation/`, вне Git; идентификаторы относятся к AniList.

На validation ALS уступала свежей популярности, а на test превосходит её.
Это разные периоды и группы пользователей; напрямую сравнивать их абсолютные
метрики как прирост качества нельзя. Период test уже исследовался в EDA,
его независимость ограничена. После этой оценки параметры не подбирались.

Исследовательский этап завершён. Следующий этап — интеграция: контракт с backend,
обучающий pipeline, сопоставление ID с каталогом сайта, запись рекомендаций,
внутренний API и сквозная проверка. Для сервиса потребуется отдельно обработать
изменения списков после обучения и пользователей, отсутствующих в ALS.
